# Lab 01-C — Tokenização e embeddings

**Skill & GO · Agentic Engineering · Aula 01 — Fundamentos**

Neste lab vemos as duas transformações que um texto sofre antes de ser usado por um modelo:

1. **Tokenização com BPE:** o texto vira uma sequência de tokens (e cada token, um número);
2. **Embeddings:** o texto vira um vetor de números que representa o seu significado, gerado por um modelo da OpenAI.

## 1. Preparar o ambiente

Instalamos o `tiktoken`, o tokenizador BPE da OpenAI, e a integração do LangChain com a OpenAI.

In [ ]:
%pip install -qU tiktoken "langchain-openai>=1.6,<2"

A tokenização roda localmente, sem chave. Já os embeddings são gerados pela API da OpenAI: no Colab, cadastre `OPENAI_API_KEY` no painel **🔑 Secrets** e habilite o acesso para este notebook.

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass  # fora do Colab: usa a variável de ambiente
except Exception as erro:
    print(f"Não foi possível ler o segredo do Colab: {erro}")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Cole sua OPENAI_API_KEY: ")

print("Chave configurada ✅")

## 2. Tokenização com BPE

Um LLM não lê letras nem palavras: ele lê **tokens**, pedaços de texto que viram números (IDs).

O **BPE** (*Byte Pair Encoding*) monta esse vocabulário durante o treinamento do tokenizador:

1. começa com os bytes do texto;
2. encontra o **par vizinho mais frequente** e o junta em um novo token;
3. repete até o vocabulário chegar ao tamanho desejado.

Resultado: palavras comuns viram um único token, e palavras raras são quebradas em pedaços conhecidos.

Usamos a codificação dos modelos GPT atuais — a mesma do `gpt-5.6-luna` dos labs anteriores.

In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-5.6-luna")
print("Codificação:", enc.name)
print("Tamanho do vocabulário:", enc.n_vocab, "tokens")

texto = "O agente consultou o manual do projeto."
ids = enc.encode(texto)
print("\nTexto:", texto)
print("IDs dos tokens:", ids)
print("De volta ao texto:", enc.decode(ids))

Agora vemos **cada pedaço**. Na saída, `␣` marca um espaço (ele costuma ficar grudado no início do token) e `�` aparece quando um caractere, como um emoji, foi dividido em mais de um token — o BPE trabalha com bytes.

In [ ]:
def mostrar_tokens(texto: str):
    ids = enc.encode(texto)
    pedacos = [enc.decode_single_token_bytes(i).decode("utf-8", errors="replace").replace(" ", "␣") for i in ids]
    print(f"{texto!r}: {len(texto)} caracteres → {len(ids)} tokens")
    print("   ", " | ".join(pedacos))


EXEMPLOS = [
    "O agente consultou o manual do projeto.",
    "Tokenização",
    "Otorrinolaringologista",
    "ISSUE-1042: POST /pricing/quote",
    "Cupom PRIMEIRA10 🚀",
]

for exemplo in EXEMPLOS:
    mostrar_tokens(exemplo)

## 3. Embeddings com modelo da OpenAI

Um **embedding** é um vetor — uma lista de números — que representa o significado de um texto. Textos com significados parecidos geram vetores próximos, e é isso que permite a busca semântica usada no RAG.

Usamos o `text-embedding-3-small`, que gera vetores de **1536 dimensões**. Ele também tokeniza o texto antes de processá-lo, com a codificação BPE dele, e aceita até 8.192 tokens por entrada.

In [ ]:
from langchain_openai import OpenAIEmbeddings

MODELO_EMBEDDING = "text-embedding-3-small"  # @param {type:"string"}

embeddings = OpenAIEmbeddings(model=MODELO_EMBEDDING)

vetor = embeddings.embed_query("Cupom de desconto aplicado duas vezes no checkout")
print("Dimensões do vetor:", len(vetor))
print("Primeiros 5 valores:", [round(valor, 4) for valor in vetor[:5]])

Textos de tamanhos diferentes têm quantidades diferentes de tokens, mas geram vetores **do mesmo tamanho**.

In [ ]:
import pandas as pd

TEXTOS = [
    "checkout",
    "Cupom de desconto aplicado duas vezes no checkout",
    "Clientes relataram que o valor final do pedido fica menor do que deveria quando voltam da etapa de endereço para a etapa de pagamento com um cupom aplicado.",
]

enc_embedding = tiktoken.encoding_for_model(MODELO_EMBEDDING)
vetores = embeddings.embed_documents(TEXTOS)

pd.DataFrame({
    "texto": TEXTOS,
    f"tokens ({enc_embedding.name})": [len(enc_embedding.encode(t)) for t in TEXTOS],
    "dimensões do vetor": [len(v) for v in vetores],
    "primeiros valores": [[round(x, 3) for x in v[:3]] for v in vetores],
})

> 💡 **Experimente:** tokenize palavras do seu dia a dia com `mostrar_tokens` e troque os `TEXTOS`.
> No Lab 01-E vamos comparar esses vetores com a similaridade de cosseno.